# XLM-R on RU22Fact: verdict words masked

Companion to the paper in [MHKhan50/multilingual-factcheck-artefacts](https://github.com/MHKhan50/multilingual-factcheck-artefacts). Same setup as the XLM-R notebook, on artefact-removed evidence from which the verdict words of each language (for example *false*, *fake*, *фейк*, *спростов*, *谣言*) are deleted. If the near-perfect Ukrainian and Russian scores come from reading the verdict, they should fall.

Kaggle settings: **GPU T4**, **Internet on**. Run all cells (about 30 minutes).

In [1]:
# Settings
REPO_URL = "https://github.com/MHKhan50/multilingual-factcheck-artefacts.git"
REPO_REF = "main"
SEEDS = "0,1,2"
VARIANTS = "cleaned_masked"
OUT = "/kaggle/working/ru22fact-neural-masked"

In [2]:
import subprocess, sys, torch
print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
import transformers; print("transformers", transformers.__version__)

torch 2.11.0+cu130 CUDA: True
GPU: Tesla T4
transformers 5.20.2


In [3]:
# Code and data (RU22Fact is downloaded from its pinned commit and checked by SHA-256)
import os
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "repo"], check=True)
subprocess.run(["git", "clone", "-q", "--branch", REPO_REF, REPO_URL, "repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-download", "--data-dir", "data/raw/ru22fact"], check=True)

1902818ba6bd0bece4f2f7e0a59cef1ac0958202

{
  "data_dir": "data/raw/ru22fact",
  "sha256": {
    "train": "9dc0f12d5d3b1a4a88a437c5630f216a31196ffd32b3ffac5d815c41d4726db8",
    "validate": "9b6e6b38b478204a28229791fde50385854b12309f2d60a49159c7f9f159f658",
    "test": "946b548f511deefac6898aad02189c809653e017aeda0f1f439def830a8a3aaf"
  }
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-download', '--data-dir', 'data/raw/ru22fact'], returncode=0)

In [4]:
# Smoke run (about a minute): catches errors before the long run
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-neural", "--data-dir", "data/raw/ru22fact",
                "--output", "/kaggle/working/smoke", "--seeds", "0", "--variants", "cleaned_masked", "--epochs", "1", "--limit", "20"], check=True)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2716.91it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "cleaned_masked", "seed": 0, "best_epoch": 1, "minutes": 0.0, "test_macro_f1": 0.33}
{
  "cleaned_masked": {
    "seeds": [
      0
    ],
    "macro_f1": {
      "mean": 0.32996632996632996,
      "std": 0.0
    },
    "per_language": {
      "en": {
        "mean": 0.3060931899641577,
        "std": 0.0
      },
      "ru": {
        "mean": 0.0,
        "std": 0.0
      },
      "uk": {
        "mean": 0.3684210526315789,
        "std": 0.0
      },
      "zh": {
        "mean": 0.3666666666666667,
        "std": 0.0
      }
    },
    "minutes": 0.0
  }
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-neural', '--data-dir', 'data/raw/ru22fact', '--output', '/kaggle/working/smoke', '--seeds', '0', '--variants', 'cleaned_masked', '--epochs', '1', '--limit', '20'], returncode=0)

In [5]:
# Full run: 3 seeds
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-neural", "--data-dir", "data/raw/ru22fact",
                "--output", OUT, "--seeds", SEEDS, "--variants", VARIANTS], check=True)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2653.86it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "cleaned_masked", "seed": 0, "best_epoch": 3, "minutes": 9.7, "test_macro_f1": 0.6784}


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2682.04it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "cleaned_masked", "seed": 1, "best_epoch": 3, "minutes": 9.9, "test_macro_f1": 0.6872}


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2756.69it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{"variant": "cleaned_masked", "seed": 2, "best_epoch": 3, "minutes": 9.7, "test_macro_f1": 0.6847}
{
  "cleaned_masked": {
    "seeds": [
      0,
      1,
      2
    ],
    "macro_f1": {
      "mean": 0.6834363761954046,
      "std": 0.004496849876372498
    },
    "per_language": {
      "en": {
        "mean": 0.6331776388526037,
        "std": 0.0031707439626318026
      },
      "ru": {
        "mean": 0.9925645781379884,
        "std": 0.006771108680420095
      },
      "uk": {
        "mean": 0.9996150885296382,
        "std": 0.0006666862230826673
      },
      "zh": {
        "mean": 0.6692685219393013,
        "std": 0.005947025101404741
      }
    },
    "minutes": 9.766666666666667
  }
}


CompletedProcess(args=['/usr/bin/python3', '-m', 'civintel_agentbench.cli', 'ru22fact-neural', '--data-dir', 'data/raw/ru22fact', '--output', '/kaggle/working/ru22fact-neural-masked', '--seeds', '0,1,2', '--variants', 'cleaned_masked'], returncode=0)

In [6]:
import json
summary = json.load(open(OUT + "/summary.json"))
for variant, s in summary.items():
    langs = ", ".join("{} {:.3f}".format(k, v["mean"]) for k, v in s["per_language"].items())
    print("{:8s} macro-F1 {:.3f} +/- {:.3f} | {} | {:.0f} min per seed".format(
        variant, s["macro_f1"]["mean"], s["macro_f1"]["std"], langs, s["minutes"]))

cleaned_masked macro-F1 0.683 +/- 0.004 | en 0.633, ru 0.993, uk 1.000, zh 0.669 | 10 min per seed
